In [74]:
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras.layers import Dense,LSTM, Dropout,Input, Concatenate
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import TimeSeriesSplit
import seaborn as sns
import matplotlib.pyplot as plt
from datetime import datetime, timedelta




# Este dataframe contiene los datos necesarios para calcular la longtud del recorrido de cada una de las lineas de subte. 

- El siguiente codigo, carga el dataset con información geográfica de las estaciones de subte de Buenos Aires y se visualizan las primeras 20 filas para una revisión inicial de su estructura.

In [75]:
#CARGO EL DATASET DONDE ME MUESTRA LAS LINEAS DE SUBTE DE LA CIUDAD DE BUENOS AIRES CON INDICADORES DE LATITUD Y LONGITUD DE CADA ESTACION 'wkt
#VISUALIZO LAS PRIMERAS 20 FILAS.
df=pd.read_csv("lineas-de-subte.csv")

df.head(20)


,wkt,id,lineasub
0,MULTILINESTRING ((-58.4521256031295 -34.566215...,1,LINEA D
1,MULTILINESTRING ((-58.4564891346516 -34.562309...,2,LINEA D
2,MULTILINESTRING ((-58.4446681474258 -34.570012...,3,LINEA D
3,MULTILINESTRING ((-58.4350135329444 -34.575178...,4,LINEA D
4,MULTILINESTRING ((-58.4257114410852 -34.578422...,5,LINEA D
5,MULTILINESTRING ((-58.4211960117166 -34.581411...,6,LINEA D
6,MULTILINESTRING ((-58.4159554193518 -34.585155...,7,LINEA D
7,MULTILINESTRING ((-58.4112939023419 -34.588237...,8,LINEA D
8,MULTILINESTRING ((-58.4071613212557 -34.591627...,9,LINEA D
9,MULTILINESTRING ((-58.3979237565356 -34.599757...,10,LINEA D


**VISUALIZACION, ANAISIS Y LIMPIEZA DE LOS DATOS**

- En el siguiente bloque de código, realizo una exploración inicial del dataset para evaluar su estructura, calidad y contenido.
- A continuación, se detallan las funciones utilizadas:
- •	df.info(): muestra el número de entradas, columnas, tipos de datos y valores no nulos por columna.
- •	df.isnull().sum(): indica la cantidad de valores nulos por columna.
- •	print(df.isna().sum().sum()): muestra el total de valores nulos en todo el dataset.
- •	df.duplicated().sum(): cuenta la cantidad de filas duplicadas.
- •	df.dtypes: informa los tipos de datos de cada columna.
- •	df.columns: lista los nombres de todas las columnas.
- •	df.sample(5): extrae 5 filas aleatorias para revisar ejemplos de datos.
- •	df.describe(): genera estadísticas descriptivas para las columnas numéricas.

- En conjunto, este análisis nos permite conocer el estado general del dataset antes de su limpieza o análisis más profundo.

In [76]:
df= df.sort_values(by='lineasub')
df.head(10)


,wkt,id,lineasub
21,MULTILINESTRING ((-58.401207533951 -34.6098335...,22,LINEA A
23,MULTILINESTRING ((-58.3984269926834 -34.609645...,24,LINEA A
22,MULTILINESTRING ((-58.392668824339 -34.6092256...,23,LINEA A
20,MULTILINESTRING ((-58.3822324012181 -34.609099...,21,LINEA A
19,MULTILINESTRING ((-58.3790851527921 -34.608881...,20,LINEA A
18,MULTILINESTRING ((-58.3742677265724 -34.608559...,19,LINEA A
48,MULTILINESTRING ((-58.436428529588 -34.6182799...,49,LINEA A
47,MULTILINESTRING ((-58.4295003242476 -34.615205...,48,LINEA A
46,MULTILINESTRING ((-58.421815670424 -34.6117702...,47,LINEA A
45,MULTILINESTRING ((-58.4151857086427 -34.610781...,46,LINEA A


In [77]:
df.info()


<class 'pandas.core.frame.DataFrame'>
Index: 82 entries, 21 to 80
Data columns (total 3 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   wkt       82 non-null     object
 1   id        82 non-null     int64 
 2   lineasub  82 non-null     object
dtypes: int64(1), object(2)
memory usage: 2.6+ KB


In [78]:
df.isnull().sum()

wkt         0
id          0
lineasub    0
dtype: int64

In [79]:
print(df.isna().sum().sum())

0


In [80]:
df.duplicated().sum()

0

In [81]:
df.dtypes



wkt         object
id           int64
lineasub    object
dtype: object

In [82]:
df.columns


Index(['wkt', 'id', 'lineasub'], dtype='object')

In [83]:
df.sample(5)



,wkt,id,lineasub
15,MULTILINESTRING ((-58.3795299795777 -34.604843...,16,LINEA C
72,MULTILINESTRING ((-58.4057948259686 -34.638405...,73,LINEA H
34,MULTILINESTRING ((-58.3807148483638 -34.603637...,35,LINEA B
1,MULTILINESTRING ((-58.4564891346516 -34.562309...,2,LINEA D
42,MULTILINESTRING ((-58.4209624717517 -34.603164...,43,LINEA B


df.shape



In [84]:
df.describe()


,id
count,82.000000
mean,41.500000
std,23.815261
min,1.000000
25%,21.250000
50%,41.500000
75%,61.750000
max,82.000000


**CON LOS DATOS 'wkt' Y LA SIGUIENTE LIBRERIA, CALCULO DE LONGITUD EN KILOMETROS DE CADA UNA DE LAS LINEAS DE SUBTE** 

In [85]:
#Libreria para convertir datos de coordenadas geograficas a KM.
from shapely import wkt

In [86]:
#GENERO UN NUEVO DATAFRAME PARA NO UTILIZAR EL ORIGINAL.
df_long = df

**Objetos Shapely**

Los objetos Shapely son una forma de representar y manipular geometrías en el espacio 2D o 3D utilizando la biblioteca Shapely en Python. Permiten crear objetos como puntos, líneas y polígonos, y realizar operaciones geométricas como intersecciones y uniones.

**Uso en el código**
En el código, se utiliza el método `apply` para convertir los valores de la columna 'wkt' en objetos Shapely, que se almacenan en la columna 'geom'.

In [87]:
# CONVIERTO LA COLUMNA 'wkt' A OBJETOS SHAPELY.
df_long['geom'] = df_long['wkt'].apply(wkt.loads)

In [88]:
# CALCULO LA LONGITUD DE LAS LINEAS.
df_long['longitud'] = df['geom'].apply(lambda x: x.length)


In [89]:
print(df_long.head(5))

                                                  wkt  id lineasub  \
21  MULTILINESTRING ((-58.401207533951 -34.6098335...  22  LINEA A   
23  MULTILINESTRING ((-58.3984269926834 -34.609645...  24  LINEA A   
22  MULTILINESTRING ((-58.392668824339 -34.6092256...  23  LINEA A   
20  MULTILINESTRING ((-58.3822324012181 -34.609099...  21  LINEA A   
19  MULTILINESTRING ((-58.3790851527921 -34.608881...  20  LINEA A   

                                                 geom  longitud  
21  MULTILINESTRING ((-58.401207533951 -34.6098335...  0.005546  
23  MULTILINESTRING ((-58.3984269926834 -34.609645...  0.002787  
22  MULTILINESTRING ((-58.392668824339 -34.6092256...  0.005773  
20  MULTILINESTRING ((-58.3822324012181 -34.609099...  0.004556  
19  MULTILINESTRING ((-58.3790851527921 -34.608881...  0.003155  


In [90]:
longitud_total_linea = df_long.groupby('lineasub')['longitud'].sum().reset_index()

In [91]:
print(longitud_total_linea)

  lineasub  longitud
0  LINEA A  0.103193
1  LINEA B  0.124921
2  LINEA C  0.041123
3  LINEA D  0.107143
4  LINEA E  0.122758
5  LINEA H  0.075421


**Tabla de equivalencias y conversion**

- En el sigiente codigo, convierto la longitud geográfica de cada línea de subte a kilómetros, utilizando la equivalencia de 1 grado decimal = 111,32 km, y se muestro la tabla resultante con las longitudes totales en kilómetros.

In [92]:
# 1 grado decimal de longitud geografica = 111.32 km.
longitud_geografica_km = 111.32

longitud_total_linea['longitud_km'] = longitud_total_linea['longitud'] * longitud_geografica_km

print(longitud_total_linea)

  lineasub  longitud  longitud_km
0  LINEA A  0.103193    11.487418
1  LINEA B  0.124921    13.906191
2  LINEA C  0.041123     4.577821
3  LINEA D  0.107143    11.927151
4  LINEA E  0.122758    13.665399
5  LINEA H  0.075421     8.395814


- Finalmente, guardo el DataFrame "longitud_total_linea" con las longitudes de las líneas de subte en kilómetros en un archivo CSV llamado "Longitud_Lineas.csv", sin incluir el índice, para su uso en futuros análisis.

In [93]:
#VISUALIZO EL NUEVO DATAFRAME CON CADA UNA DE LAS LINEAS Y SU LONGITUD DE RECORRIDO EN KM.

ltl = longitud_total_linea

In [94]:
ltl

,lineasub,longitud,longitud_km
0,LINEA A,0.103193,11.487418
1,LINEA B,0.124921,13.906191
2,LINEA C,0.041123,4.577821
3,LINEA D,0.107143,11.927151
4,LINEA E,0.122758,13.665399
5,LINEA H,0.075421,8.395814


In [95]:
#DECIDO GRABAR EL NUEVO DATAFRAME EN UN NUEVO ARCHIVO PRA FUTUROS ANALISIS.
ltl.to_csv('Longitud_Lineas.csv', index=False)